# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [1]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

### Total Number of Flows

Count the total number of flows in this trace.

In [18]:
num_flows = ndf.groupby(['Source', 'Destination'])
num_flows.size()

Source                     Destination          
0.0.0.0                    255.255.255.255            8
                           all-systems.mcast.net      4
104.31.113.215             192.168.43.72              6
17.188.166.20              192.168.43.72              2
17.252.44.15               192.168.43.72              3
                                                   ... 
par10s29-in-f3.1e100.net   192.168.43.72              8
par10s29-in-f4.1e100.net   192.168.43.72             79
par10s38-in-f13.1e100.net  192.168.43.72             30
par10s38-in-f3.1e100.net   192.168.43.72            144
par21s03-in-f2.1e100.net   192.168.43.72             15
Length: 77, dtype: int64

50


### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [19]:
flow_bytes = (
    ndf.groupby('flow_id')['Length']
    .sum()
    .reset_index(name='total_bytes')
    .sort_values('total_bytes', ascending=False)
)
flow_bytes.head(20)

,flow_id,total_bytes
22,"(192.168.43.72, ipv4-c071-cdg001-ix.1.oca.nflx...",123964470
21,"(192.168.43.72, ipv4-c069-cdg001-ix.1.oca.nflx...",7382603
14,"(192.168.43.72, a23-57-80-120.deploy.static.ak...",1392549
16,"(192.168.43.72, ec2-52-19-39-146.eu-west-1.com...",688471
20,"(192.168.43.72, ipv4-c063-cdg001-ix.1.oca.nflx...",454339
10,"(192.168.43.72, 198.38.120.137)",123361
30,"(192.168.43.72, par10s38-in-f3.1e100.net)",116099
24,"(192.168.43.72, ipv4-c197-cdg001-ix.1.oca.nflx...",94895
17,"(192.168.43.72, ec2-52-208-128-101.eu-west-1.c...",93172
28,"(192.168.43.72, par10s29-in-f4.1e100.net)",78383


### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [20]:
flow_bytes = (
    ndf.groupby('flow_id')
    .size()
    .reset_index(name='total_packages')
    .sort_values('total_packages', ascending=False)
)
flow_bytes.head(20)

,flow_id,total_packages
22,"(192.168.43.72, ipv4-c071-cdg001-ix.1.oca.nflx...",127986
21,"(192.168.43.72, ipv4-c069-cdg001-ix.1.oca.nflx...",8043
14,"(192.168.43.72, a23-57-80-120.deploy.static.ak...",1839
16,"(192.168.43.72, ec2-52-19-39-146.eu-west-1.com...",961
20,"(192.168.43.72, ipv4-c063-cdg001-ix.1.oca.nflx...",604
30,"(192.168.43.72, par10s38-in-f3.1e100.net)",302
17,"(192.168.43.72, ec2-52-208-128-101.eu-west-1.c...",191
10,"(192.168.43.72, 198.38.120.137)",190
28,"(192.168.43.72, par10s29-in-f4.1e100.net)",151
11,"(192.168.43.72, 198.38.120.153)",138


### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [10]:
# Make sure Time is a proper datetime type
ndf['Time'] = pd.to_datetime(ndf['Time'])

flow_duration = (
    ndf.groupby('flow_id')['Time']
    .agg(lambda x: x.max() - x.min())
    .reset_index(name='duration')
    .sort_values('duration', ascending=False)
)
flow_duration.head(20)

,flow_id,duration
71,"(192.168.43.72, par10s38-in-f3.1e100.net, TCP)",0 days 00:08:15.451688
58,"(192.168.43.72, ns-vip-pro.paris.inria.fr, DNS)",0 days 00:08:13.065143
72,"(192.168.43.72, par10s38-in-f3.1e100.net, TLSv...",0 days 00:08:12.823938
70,"(192.168.43.72, par10s38-in-f3.1e100.net, SSL)",0 days 00:08:10.686842
76,"(192.168.43.97, 224.0.0.251, MDNS)",0 days 00:08:06.705466
21,"(192.168.43.72, 224.0.0.251, MDNS)",0 days 00:08:06.400795
94,"(fe80::e6ce:8fff:fe01:4c54, ff02::fb, MDNS)",0 days 00:08:06.400733
27,"(192.168.43.72, ec2-34-252-77-54.eu-west-1.com...",0 days 00:08:00.552744
28,"(192.168.43.72, ec2-34-252-77-54.eu-west-1.com...",0 days 00:08:00.407570
46,"(192.168.43.72, ipv4-c069-cdg001-ix.1.oca.nflx...",0 days 00:07:55.487773


## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?